# Cell 1 — Markdown
# Step 6: Baseline Model Development

## Objective

The objective of this step is to build baseline machine learning models that predict whether a customer will respond to a commercial campaign.

The target variable is:

**y = customer campaign response**

In the UCI Bank Marketing dataset:

- `yes` means the customer subscribed to the term deposit
- `no` means the customer did not subscribe

This step compares three baseline models:

| Model | Purpose |
|---|---|
| Logistic Regression | Interpretable baseline model |
| Random Forest | Nonlinear model that captures complex patterns |
| Gradient Boosting | Strong predictive model for campaign response prediction |

The final recommended model will be selected based on predictive performance and business usefulness.

In [1]:
# Cell 2 — Import Libraries
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Cell 3 — Locate Project Folder and Load Data
# Find project root folder
current_dir = Path.cwd()

if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError("Could not find the data folder. Make sure you are working inside the project repository.")

# Dataset path
data_path = PROJECT_ROOT / "data" / "raw" / "bank-additional-full.csv"

# Load dataset
df = pd.read_csv(data_path, sep=";")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
# Cell 4 — Prepare Target Variable
# Check target values
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True) * 100)

y
no     36548
yes     4640
Name: count, dtype: int64
y
no     88.734583
yes    11.265417
Name: proportion, dtype: float64


In [4]:
# Cell 4 — Prepare Target Variable
# Convert target variable to binary format
# yes = 1 means customer responded
# no = 0 means customer did not respond

df["target"] = df["y"].map({"yes": 1, "no": 0})

# Drop original target column
df_model = df.drop(columns=["y"])

df_model.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,target
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0


In [5]:
# Cell 5 — Define Features and Target
X = df_model.drop(columns=["target"])
y = df_model["target"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (41188, 20)
Target shape: (41188,)


In [6]:
# Cell 6 — Identify Numerical and Categorical Variables
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


In [7]:
# Cell 7 — Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining response rate:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest response rate:")
print(y_test.value_counts(normalize=True) * 100)


Training set shape: (32950, 20)
Test set shape: (8238, 20)

Training response rate:
target
0    88.734446
1    11.265554
Name: proportion, dtype: float64

Test response rate:
target
0    88.73513
1    11.26487
Name: proportion, dtype: float64


In [8]:
# Cell 8 — Build Preprocessing Pipeline
# Version-compatible OneHotEncoder
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder)
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [9]:
# Cell 9 — Define Baseline Models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.10,
        max_depth=3,
        random_state=42
    )
}

models

{'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
 'Random Forest': RandomForestClassifier(class_weight='balanced', min_samples_leaf=5,
                        n_estimators=300, n_jobs=-1, random_state=42),
 'Gradient Boosting': GradientBoostingClassifier(random_state=42)}

In [10]:
# Cell 10 — Train and Evaluate Models
results = []

trained_models = {}

for model_name, model in models.items():
    print(f"Training model: {model_name}")
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_prob),
        "PR AUC": average_precision_score(y_test, y_prob)
    }
    
    results.append(metrics)
    trained_models[model_name] = pipeline

print("Model training completed.")

Training model: Logistic Regression
Training model: Random Forest
Training model: Gradient Boosting
Model training completed.


In [11]:
# Cell 11 — Compare Model Results
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["ROC AUC", "PR AUC"],
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,PR AUC
2,Gradient Boosting,0.921704,0.695712,0.542026,0.609328,0.953481,0.691556
1,Random Forest,0.889172,0.504559,0.894397,0.645161,0.952949,0.692567
0,Logistic Regression,0.865137,0.451200,0.911638,0.603639,0.943838,0.622248


In [12]:
# Cell 12 — Save Model Comparison Results
reports_path = PROJECT_ROOT / "reports"
reports_path.mkdir(exist_ok=True)

results_file = reports_path / "baseline_model_results.csv"

results_df.to_csv(results_file, index=False)

print(f"Baseline model results saved to: {results_file}")

Baseline model results saved to: c:\Users\nejat\OneDrive\Desktop\UN\Skills\GitHub 2026\Commercial Campaign Response Prediction\commercial-campaign-response-prioritization\reports\baseline_model_results.csv


In [13]:
# Cell 13 — Select Best Model
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best baseline model:")
print(best_model_name)

results_df.iloc[0]

Best baseline model:
Gradient Boosting


Model        Gradient Boosting
Accuracy              0.921704
Precision             0.695712
Recall                0.542026
F1 Score              0.609328
ROC AUC               0.953481
PR AUC                0.691556
Name: 2, dtype: object

In [14]:
# Cell 14 — Detailed Evaluation of Best Model
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred_best))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))


Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      7310
           1       0.70      0.54      0.61       928

    accuracy                           0.92      8238
   macro avg       0.82      0.76      0.78      8238
weighted avg       0.92      0.92      0.92      8238

Confusion Matrix:
[[7090  220]
 [ 425  503]]


#Cell 15 — Markdown Interpretation
## Baseline Model Results Summary

Three baseline models were trained and evaluated:

1. Logistic Regression
2. Random Forest
3. Gradient Boosting

Because the campaign-response problem is imbalanced, accuracy alone is not enough to evaluate model quality. The most important metrics are:

- **ROC AUC**, because it measures the model's ability to rank responders above non-responders
- **PR AUC**, because it is useful when the positive response class is relatively small
- **Recall**, because the business wants to identify as many likely responders as possible
- **Precision**, because outreach resources should not be wasted on customers unlikely to respond

The best baseline model should be selected based on ranking ability and commercial usefulness, not only raw accuracy.

For this project, the recommended final model candidate is the model with the strongest ROC AUC and PR AUC performance. In most campaign-response problems, either **Gradient Boosting** or **Random Forest** is expected to perform better than Logistic Regression because these models can capture nonlinear customer behavior patterns.

In [15]:
# Cell 16 — Save Customer-Level Prediction Scores
scored_customers = X_test.copy()

scored_customers["actual_response"] = y_test.values
scored_customers["predicted_response_probability"] = y_prob_best
scored_customers["predicted_response"] = y_pred_best

scored_customers = scored_customers.sort_values(
    by="predicted_response_probability",
    ascending=False
)

processed_path = PROJECT_ROOT / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

scores_file = processed_path / "baseline_customer_response_scores.csv"

scored_customers.to_csv(scores_file, index=False)

print(f"Customer response scores saved to: {scores_file}")

scored_customers.head(10)

Customer response scores saved to: c:\Users\nejat\OneDrive\Desktop\UN\Skills\GitHub 2026\Commercial Campaign Response Prediction\commercial-campaign-response-prioritization\data\processed\baseline_customer_response_scores.csv


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,actual_response,predicted_response_probability,predicted_response
39655,92,retired,married,unknown,no,yes,no,cellular,may,thu,...,2,success,-1.8,93.876,-40.0,0.683,5008.7,0,0.933989,1
39498,80,retired,divorced,basic.4y,no,yes,yes,cellular,apr,thu,...,2,success,-1.8,93.749,-34.6,0.644,5008.7,1,0.924175,1
40078,72,retired,married,basic.4y,no,no,yes,cellular,jul,fri,...,1,success,-1.7,94.215,-40.3,0.822,4991.6,1,0.918346,1
40450,92,retired,married,unknown,no,no,yes,cellular,aug,tue,...,1,success,-1.7,94.027,-38.3,0.904,4991.6,1,0.918231,1
39993,27,unknown,single,university.degree,no,yes,no,cellular,jun,wed,...,2,success,-1.7,94.055,-39.8,0.767,4991.6,1,0.917278,1
29052,19,student,single,unknown,no,yes,no,telephone,apr,fri,...,0,nonexistent,-1.8,93.075,-47.1,1.405,5099.1,0,0.913175,1
39153,21,student,single,basic.9y,no,no,no,cellular,mar,wed,...,1,success,-1.8,93.369,-34.8,0.655,5008.7,1,0.911803,1
39680,52,management,married,university.degree,no,yes,no,cellular,may,fri,...,3,success,-1.8,93.876,-40.0,0.684,5008.7,1,0.910231,1
39607,52,admin.,married,university.degree,no,no,no,cellular,may,thu,...,2,success,-1.8,93.876,-40.0,0.677,5008.7,1,0.910172,1
40713,30,technician,single,professional.course,no,no,no,cellular,sep,tue,...,1,success,-1.1,94.199,-37.5,0.877,4963.6,0,0.907785,1
